In [ ]:
# ACCOUNT B — DEPLOY / SUBMISSION STABLE (frozen)
# Pipeline: Type1 = v40.5 MC-only symbolic prehandler -> LoRA + v35 fallback ; Type2 = solver.
# DO NOT: finetune, augment, add new rules, try semantic alias, restart tunnel during judging.
# Promote a new engine ONLY after Account C shadow_eval returns PROMOTE.
RUN_SERVE=True; RUN_API=True; RUN_TUNNEL=True
BASE_MODEL="/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
TYPE1_LORA="/kaggle/input/datasets/nguyenminhtric/exact-test/DATASET_DATA_TYPE_1"
TYPE1_PATCH="/kaggle/input/datasets/nguyenminhtric/exact-test/type1_fixed_patch/type1_fixed"
VLLM_HOST="127.0.0.1"; VLLM_PORT=8001; API_PORT=9000
BASE_SERVED_NAME="qwen3-8b-base"; LORA_SERVED_NAME="qwen3-8b-exact-type1"
MAX_MODEL_LEN=4096; GEN_MAX_TOKENS=256
print("ACCOUNT B deploy (frozen baseline = v40.5 MC-only)")

In [ ]:
# Install vLLM (proven ladder, pin 0.22.1)
import os,sys,subprocess
os.environ["TRANSFORMERS_NO_TF"]="1"; os.environ["USE_TF"]="0"; os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"]="python"
def _pip(*a): return subprocess.run([sys.executable,"-m","pip","install","--quiet","--break-system-packages"]+list(a),capture_output=True,text=True)
def _clear():
    for m in list(sys.modules):
        if any(x in m for x in ("vllm","transformers","torch._dynamo","torch._inductor")): del sys.modules[m]
def _imp():
    try:
        from vllm import LLM; return True,""
    except Exception as e: return False,str(e).split("\n")[0][:160]
ok,_=_imp()
if not ok:
    for tv,vv in [("4.48.0","0.22.1"),("4.44.2","0.22.1"),("4.57.6","0.22.1"),("4.48.0","")]:
        pkg=f"vllm=={vv}" if vv else "vllm"; _pip("protobuf==5.29.5"); _pip(f"transformers=={tv}"); _pip(pkg); _clear(); ok,_=_imp()
        if ok: break
_pip("xformers","fastapi","uvicorn","requests","httptools")
import vllm,torch; print("vllm",vllm.__version__,"gpus",torch.cuda.device_count())

In [ ]:
# T4 env: TRITON_ATTN (FlashInfer crashes on T4)
import os,subprocess
subprocess.run("mkdir -p /usr/local/cuda/lib64 && ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /usr/local/cuda/lib64/libcuda.so",shell=True)
os.environ.update({"CUDA_VISIBLE_DEVICES":"0,1","VLLM_USE_V1":"0","VLLM_ATTENTION_BACKEND":"TRITON_ATTN",
  "VLLM_WORKER_MULTIPROC_METHOD":"spawn","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True",
  "LD_LIBRARY_PATH":"/usr/local/cuda/lib64:/usr/local/nvidia/lib64:"+os.environ.get("LD_LIBRARY_PATH","")})
import json
try: LORA_RANK=int(json.load(open(os.path.join(TYPE1_LORA,"adapter_config.json"))).get("r",16))
except Exception: LORA_RANK=16
MAX_LORA_RANK=next(x for x in [8,16,32,64,128] if x>=LORA_RANK); print("backend TRITON_ATTN, max_lora_rank",MAX_LORA_RANK)

In [ ]:
# Launch vLLM OpenAI server (TP=2, dynamic LoRA)
import sys,time,subprocess,pathlib,requests
if RUN_SERVE:
    subprocess.run("pkill -f vllm || true",shell=True); time.sleep(5)
    log="/kaggle/working/vllm.log"; lf=open(log,"w")
    cmd=[sys.executable,"-m","vllm.entrypoints.openai.api_server","--model",BASE_MODEL,"--served-model-name",BASE_SERVED_NAME,
      "--dtype","float16","--tensor-parallel-size","2","--max-model-len",str(MAX_MODEL_LEN),"--gpu-memory-utilization","0.85",
      "--max-num-seqs","1","--enforce-eager","--disable-custom-all-reduce","--no-enable-prefix-caching","--no-enable-chunked-prefill","--generation-config","vllm",
      "--enable-lora","--max-lora-rank",str(MAX_LORA_RANK),"--lora-modules",f"{LORA_SERVED_NAME}={TYPE1_LORA}","--host",VLLM_HOST,"--port",str(VLLM_PORT)]
    p=subprocess.Popen(cmd,stdout=lf,stderr=lf,env=os.environ.copy()); print("pid",p.pid); ready=False
    for i in range(60):
        time.sleep(10)
        try:
            if requests.get(f"http://{VLLM_HOST}:{VLLM_PORT}/v1/models",timeout=5).status_code==200: print("vLLM READY"); ready=True; break
        except Exception: pass
        if p.poll() is not None: print(pathlib.Path(log).read_text()[-5000:]); break
        print("wait",i)
    if not ready: raise RuntimeError("vLLM not ready (try VLLM_ATTENTION_BACKEND=FLEX_ATTENTION)")

In [ ]:
# verifier_v35 from patch
from pathlib import Path
import shutil,sys,types,importlib.util
WORK=Path("/kaggle/working/exact_runtime"); shutil.rmtree(WORK,ignore_errors=True); (WORK/"app/type1_logic").mkdir(parents=True,exist_ok=True)
for p in [WORK/"app/__init__.py",WORK/"app/type1_logic/__init__.py"]: p.write_text("")
for n in ["parser.py","prompt.py","solver.py","verifier_v35.py"]: shutil.copy(Path(TYPE1_PATCH)/n,WORK/"app/type1_logic"/n)
am=types.ModuleType("app"); am.__path__=[str(WORK/"app")]; sys.modules["app"]=am
tm=types.ModuleType("app.type1_logic"); tm.__path__=[str(WORK/"app/type1_logic")]; sys.modules["app.type1_logic"]=tm
for n in ["parser","prompt"]:
    sp=importlib.util.spec_from_file_location(f"app.type1_logic.{n}",WORK/"app/type1_logic"/f"{n}.py"); m=importlib.util.module_from_spec(sp); sys.modules[f"app.type1_logic.{n}"]=m; sp.loader.exec_module(m)
sp=importlib.util.spec_from_file_location("app.type1_logic.verifier_v35",WORK/"app/type1_logic/verifier_v35.py"); V35=importlib.util.module_from_spec(sp); sys.modules["app.type1_logic.verifier_v35"]=V35; sp.loader.exec_module(V35)
def call_v35(q,prem,ans): return V35.verify(q,prem,ans)
print("verifier_v35 loaded")

In [ ]:
# --- v40 entity-conjunctive engine (frozen baseline; symbolic prehandler) ---
import re
# -*- coding: utf-8 -*-
"""v40.3 entity-grounded conjunctive Horn engine for the REAL Phase-1 distribution.
Propositional over a single entity: facts = literals, rules = (conj of literals)->literal.
Forward-chain; answer options by derivability. Certificate = premise indices. Abstain-safe."""
import re
STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','its','it','they','them',
      'is','are','was','were','be','been','has','have','had','then','if','no','not','with','as','by','from',
      'artifact','package','manuscript','sample','batch','item','device','record','file','student'}
def _stem(t):
    if re.search(r'(ss|us|is)$',t): return t
    if re.search(r'(ches|shes|xes|zes|ses)$',t): return t[:-2]
    if re.search(r'ies$',t): return t[:-3]+'y'
    if t.endswith('s'): t=t[:-1]
    return re.sub(r'(ing|ed)$','',t)
def atom_key(phrase):
    s=re.sub(r'(?<!^)(?=[A-Z])',' ',str(phrase)).lower()
    nums=re.findall(r'\d+', s)
    toks=[ _stem(w) for w in re.findall(r'[a-zA-Z]+', s) ]
    toks=[t for t in toks if t and t not in STOP and len(t)>2]
    keys=sorted(set(toks))+["N"+n for n in sorted(set(nums))]   # keep numeric literals
    return "".join(w.capitalize() for w in keys) if keys else None

# split a clause into (atom, neg). Handles "X has Y", "X is Y", "X has no Y", "X lacks Y", "cannot ...", "is not ..."
_LEAD=re.compile(r"^\s*(if|then|that|who|which|it|its|their|this)\b",re.I)
_VERB=re.compile(r"\b(cannot|can not|can|could|may|might|must|should|shall|will|would|is not|are not|was not|were not|isn't|aren't|is|are|was|were|has no|have no|had no|has|have|had|lacks?|without|requires?|needs?|contains?|completed?|enters?|gains?|receives?|provides?|shows?|states?|holds?|carries|monitors?|captures?|eligible|allowed|approved|assigned|be|been|being)\b",re.I)
ACTION_VERBS={'receives','receive','provides','provide','shows','show','states','state','monitors','monitor','captures','capture','enters','enter','requires','require','needs','need','gains','gain','completed','complete','contains','contain','reports','report','releases','release','passes','pass','improves','improve','supports','support','recommends','recommend','administered','administer','approved','approve'}
_NEGWORD=re.compile(r'\b(no|not|cannot|can not|lacks?|without|isn\'t|aren\'t|never|nor|incomplete|missing|lacking)\b',re.I)
def to_literal(clause):
    c=clause.strip().rstrip('.').strip()
    c=_LEAD.sub('',c).strip()
    neg=bool(re.search(r"\b(no|not|cannot|can not|never|lacks?|without|isn't|aren't|incomplete|missing|lacking|nor|un(?:able|verified|established))\b",c,re.I))
    m=_VERB.search(c)
    pred=c[m.end():].strip() if m else c
    verb=(m.group(1).lower() if m else '')
    if m and verb in ACTION_VERBS:
        pred = verb + ' ' + pred
    # peel any leftover leading modal/aux/passive markers and articles
    for _ in range(4):
        pred=re.sub(r"^\s*(be|been|being|to|a|an|the|no|not|its|their)\b","",pred,flags=re.I).strip()
    a=atom_key(pred)
    return (a,neg) if a else None

def parse_premise(p):
    s=str(p).strip()
    m=re.search(r'^\s*if\b(.+?),?\s*\bthen\b(.+)$',s,re.I)
    if m:
        ante=re.split(r'\band\b',m.group(1),flags=re.I)
        lits=[to_literal(x) for x in ante]; lits=[l for l in lits if l]
        con=to_literal(m.group(2))
        if con and lits: return ('rule',lits,con)
        return None
    # Universal relative rule: Every/All <role> <condition> <verb> <consequent>
    # Examples: Every volunteer assigned to shift receives badge; All satellites with cameras can capture images.
    m2=re.search(r'^\s*(every|all)\s+[a-zA-Z]+s?\s+(.+?)\s+\b(can|may|must|should|will|would|receives?|gets?|gains?|provides?|captures?|monitors?|requires?|needs?|is|are)\b\s+(.+)$',s,re.I)
    if m2:
        cond=m2.group(2).strip()
        cons=(m2.group(3)+" "+m2.group(4)).strip()
        litc=to_literal(cond); litd=to_literal(cons)
        if litc and litd: return ('rule',[litc],litd)
    if re.search(r'^\s*(no premise|it (is|cannot)|unknown|there is no information)',s,re.I): return None
    lit=to_literal(s)
    return ('fact',lit) if lit else None

def solve_entity(premises):
    facts={}  # atom -> (bool_value, premise_idx)
    rules=[]
    for i,p in enumerate(premises):
        pp=parse_premise(p)
        if not pp: continue
        if pp[0]=='fact':
            a,neg=pp[1]; facts.setdefault(a,(not neg,[i]))
        else:
            rules.append((i,pp[1],pp[2]))
    changed=True
    while changed:
        changed=False
        for i,lits,con in rules:
            ca,cneg=con
            ok=True; path=[i]
            for a,neg in lits:
                if a in facts and facts[a][0]==(not neg): path+=facts[a][1]
                else: ok=False; break
            if ok and ca not in facts:
                facts[ca]=((not cneg),sorted(set(path))); changed=True
    return facts

_META_RE=__import__("re").compile(r"\b(not (?:yet )?(?:established|confirmed|verified|approved|cleared|determined)|cannot be (?:established|confirmed)|unsupported|is not established|no premise|undetermined|not (?:available|present))\b",__import__("re").I)
def decompose_option(opt):
    import re
    t=re.sub(r'^\s*[A-Da-d][.):]\s*','',str(opt)).strip()
    t=re.split(r'\bbecause\b', t, maxsplit=1, flags=re.I)[0].strip()  # drop causal justification
    parts=re.split(r',\s*but\s+|\s+but\s+|;\s+|\s+while\s+|\s+whereas\s+|\s+and\s+',t,flags=re.I)
    claims=[]
    for p in parts:
        p=p.strip()
        if not p: continue
        is_meta=bool(_META_RE.search(p))
        lit=to_literal(p)
        claims.append((lit,is_meta,p))
    return claims

def answer_mc(premises, options):
    facts=solve_entity(premises)
    res={}
    for lab,opt in zip("ABCD",options):
        claims=decompose_option(opt)
        if not claims: res[lab]=('UNSUP',[]); continue
        status='PROVEN'; path=[]
        for lit,is_meta,txt in claims:
            if lit is None: status='UNSUP'; break
            a,neg=lit; have = a in facts
            val = facts[a][0] if have else None
            if is_meta:
                # meta "not established": correct only if NOT positively proven
                if have and val==True: status='DISPROVEN'; break
            else:
                if have and val==(not neg): path+=facts[a][1]
                elif have and val==(neg): status='DISPROVEN'; break
                else: status='UNSUP'; break
        res[lab]=(status, sorted(set(path)))
    proven=[l for l in res if res[l][0]=='PROVEN']
    if len(proven)==1: return proven[0],res[proven[0]][1],'entity_unique_proof',res
    return None,[],('multiple' if proven else 'none'),res


# ================= Phase-1 REPLAY HARNESS =================
def _opt_texts(rp):
    import re
    f=[o[1].strip().replace("\n"," ") for o in re.findall(r"(?:^|\n)\s*([A-D])[.)]\s*(.+?)(?=\n\s*[A-D][.)]|\Z)",rp.get("query",""),flags=re.S)]
    return f if len(f)>=2 else (rp.get("options") or [])
def replay_phase1(path):
    import json,re
    d=json.load(open(path)); t1=[l for l in d["logs"] if l.get("type")=="type1"]
    fired=aok=pok=0; fixed=[]; wrong=[]; abst=0
    for l in t1:
        rp=l["request_payload"]; exp=l.get("expected") or {}; ea=str(exp.get("answer","")).strip().upper()
        opts=_opt_texts(rp)
        if not opts: continue
        a,pu,why,res=answer_mc(rp.get("premises",[]) or [], opts)
        if a is None: abst+=1; continue
        fired+=1; ok=(a==ea); aok+=ok; pok+=(sorted(pu)==sorted(exp.get("premises_used") or []))
        if ok and l.get("status")!="correct": fixed.append(l["query_id"])
        if not ok: wrong.append({"query_id":l["query_id"],"exp":ea,"got":a})
    rep={"n":len(t1),"fired":fired,"answer_correct":aok,"premises_used_correct":pok,
         "wrong":wrong,"fixed_old_wrong":fixed,"abstained":abst,
         "precision_when_fired":round(aok/max(fired,1),3),"coverage":round(fired/max(len(t1),1),3),
         "gate":"ABSTAIN_SAFE" if not wrong else "HAS_WRONG_FIX_BEFORE_APPLY"}
    return rep
def _autofind():
    import glob,os,sys
    if len(sys.argv)>1 and os.path.exists(sys.argv[1]): return sys.argv[1]
    for c in ["exact_eval_round1_Astatine.json","/kaggle/input/**/exact_eval_round1_Astatine.json",
              "/kaggle/working/exact_eval_round1_Astatine.json","./exact_eval_round1_Astatine.json"]:
        h=sorted(glob.glob(c,recursive=True)) if any(x in c for x in "*?[") else ([c] if os.path.exists(c) else [])
        if h: return h[0]
    return None
print('v40 symbolic engine loaded')

In [ ]:
# --- ANSWER-AWARE premises_used proof-certificate (v2) ---
# Phản biện chồng (2026-06-20): cert PHẢI gắn với answer đang trả, không union mọi option.
#   MC (letter A-D hoặc option-text): chỉ trace premises của ĐÚNG option được trả.
#   YNU (Yes/No/Unknown/Uncertain): KHÔNG đụng -> giữ old premises_used.
#   Guards: proof_trace only; len(cert)>=len(old); cap MAX_CERT_LEN. Never change answer.
MAX_CERT_LEN = 12
def _resolve_option_index(answer, opts):
    a=str(answer).strip()
    if re.fullmatch(r"[A-Da-d]", a):
        i="ABCD".index(a.upper()); return i if i<len(opts) else None
    if a.lower() in {"yes","no","unknown","uncertain",""}:
        return None
    norm=lambda s: re.sub(r"[^a-z0-9]","",str(s).lower())
    na=norm(a)
    if not na: return None
    for i,o in enumerate(opts):
        if norm(o)==na: return i
    for i,o in enumerate(opts):
        if na and (na in norm(o) or norm(o) in na): return i
    return None
def premises_used_certificate(answer, question, premises, options):
    opts=_opt_texts(question) or options
    if not opts: return [], "no_options"
    i=_resolve_option_index(answer, opts)
    if i is None: return [], "skip_ynu"
    atoms=[lit[0] for lit,_m,_t in decompose_option(opts[i]) if lit]
    if not atoms: return [], "no_atom"
    try: facts=solve_entity(premises)
    except TypeError: facts=solve_entity(premises,None)
    used=set(); proven=False
    for at in atoms:
        if at in facts: used|=set(facts[at][1]); proven=True
    return (sorted(used),"answer_option_proof_trace") if proven else ([], "answer_not_derivable")
def apply_premises_certificate(premises_used, answer, question, premises, options):
    old=list(premises_used or [])
    c,src=premises_used_certificate(answer, question, premises, options)
    if src!="answer_option_proof_trace": return old
    if not c: return old
    if len(c)<len(old): return old
    if len(c)>MAX_CERT_LEN: return old
    return c
print("premises_used ANSWER-AWARE certificate ready (MC-only, proof-trace, cap=%d)"%MAX_CERT_LEN)


In [ ]:
# generation helper (vLLM) + parser
import re,time,requests
VURL=f"http://{VLLM_HOST}:{VLLM_PORT}/v1/completions"
def _opt_texts(q):
    return [o[1].strip().replace("\n"," ") for o in re.findall(r"(?:^|\n)\s*([A-D])[.)]\s*(.+?)(?=\n\s*[A-D][.)]|\Z)",q,flags=re.S)]
def generate_vllm(prompt,mx=GEN_MAX_TOKENS):
    r=requests.post(VURL,json={"model":LORA_SERVED_NAME,"prompt":prompt,"max_tokens":mx,"temperature":0.0,"top_p":1.0,
      "stop":["\nReasoning:","\n\nReasoning:","\nQuestion:","\n\nQuestion:","\n###"]},timeout=55); r.raise_for_status()
    return r.json()["choices"][0]["text"]
def parse_final(t):
    m=re.findall(r"Final Answer:\s*([A-D]|Yes|No|Unknown|Uncertain)\b",t or "",re.I)
    if not m: return "Unknown"
    v=m[0].strip(); v=v.upper() if re.fullmatch(r"[A-Da-d]",v) else v.title(); return "Unknown" if v=="Uncertain" else v
print("gen + parser ready")

In [ ]:
# /predict pipeline: v40 symbolic prehandler -> LoRA + v35 fallback ; Type2 deterministic
def _field(req,n,d=None):
    return (req.get(n,d) if isinstance(req,dict) else getattr(req,n,d))
def symbolic_prehandler_type1(req):
    q=_field(req,"query","") or ""; premises=list(_field(req,"premises",[]) or []); options=list(_field(req,"options",[]) or [])
    opt_texts=_opt_texts(q) or options
    if not opt_texts or set(str(o).strip().lower() for o in options)<= {"yes","no","unknown","uncertain"}:
        return None  # YNU not covered by frozen baseline
    a,pu,why,res=answer_mc(premises,opt_texts)
    if a is None: return None
    ans=a
    if re.fullmatch(r"[A-D]",str(a) or "") and options and a not in options and not _opt_texts(q):
        i="ABCD".index(a); ans=options[i] if i<len(options) else a
    return {"query_id":_field(req,"query_id","unknown"),"answer":ans,"unit":"",
            "explanation":f"Derived by symbolic proof ({why}).","premises_used":pu,
            "reasoning":{"source":"v40_5_symbolic","rule":why}}
def predict_type1_lora(req):
    q=_field(req,"query","") or ""; premises=list(_field(req,"premises",[]) or []); options=list(_field(req,"options",[]) or [])
    lines=["You are solving a logic-based educational QA problem. Use only the given premises. Do not use outside knowledge.","","Premises:"]
    for i,p in enumerate(premises,1): lines.append(f"{i}. {p}")
    lines+=["","Question:",q,"","Reason step by step briefly, cite supporting premises if useful, and End with exactly one line: Final Answer: <Yes, No, or one option>"]
    raw=generate_vllm("\n".join(lines)+"\n"); ans=parse_final(raw)
    try: va,vp,vr=call_v35(q,premises,ans)
    except Exception: va,vp,vr=None,[],"verifier_error"
    if va is not None: ans=va; pu=vp or []
    else: pu=[]
    pu=apply_premises_certificate(pu, ans, q, premises, options)   # ANSWER-AWARE certificate override (live-safe, MC-only)
    if options and ans not in options:
        if ans=="Unknown" and "Uncertain" in options: ans="Uncertain"
        elif ans not in options: ans = options[0] if options else ans
    return {"query_id":_field(req,"query_id","unknown"),"answer":ans,"unit":"","explanation":"LoRA + v35 fallback.","premises_used":pu,"reasoning":{"source":"lora_v35"}}
def predict_type1(req):
    s=symbolic_prehandler_type1(req)
    return s if s is not None else predict_type1_lora(req)
def solve_type2(req):
    qid=_field(req,"query_id","unknown"); ql=(_field(req,"query","") or "").lower()
    nums=re.findall(r"r\d*\s*=\s*([0-9.]+)\s*ohm",ql); v=re.search(r"([0-9.]+)\s*v\b",ql)
    if "parallel" in ql and "current" in ql and len(nums)>=2 and v:
        rs=[float(x) for x in nums[:2]]; V=float(v.group(1)); I=sum(V/r for r in rs); a=str(int(round(I))) if abs(I-round(I))<1e-9 else f"{I:.6g}"
        return {"query_id":qid,"answer":a,"unit":"A","explanation":"deterministic solver","premises_used":[],"reasoning":None}
    return {"query_id":qid,"answer":"0","unit":"","explanation":"Type2 fallback","premises_used":[],"reasoning":None}
print("predict pipeline ready (symbolic -> LoRA+v35 -> Type2)")

In [ ]:
# FastAPI + proxy + tunnel
from fastapi import FastAPI,Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel,Field
from typing import Literal
import uvicorn,threading,time,requests
VB=f"http://{VLLM_HOST}:{VLLM_PORT}"
class PR(BaseModel):
    query_id:str; type:Literal["type1","type2"]; query:str; premises:list[str]=Field(default_factory=list); options:list[str]=Field(default_factory=list)
app=FastAPI()
@app.get("/health")
def h():
    try: return {"status":"ok","vllm":requests.get(f"{VB}/v1/models",timeout=5).status_code}
    except Exception as e: return {"status":"degraded","err":type(e).__name__}
@app.get("/v1/models")
def pm(): r=requests.get(f"{VB}/v1/models",timeout=10); return JSONResponse(r.json(),status_code=r.status_code)
@app.post("/v1/completions")
async def pc(req:Request): r=requests.post(f"{VB}/v1/completions",json=await req.json(),timeout=55); return JSONResponse(r.json(),status_code=r.status_code)
@app.post("/predict")
def predict(req:PR):
    try: return [solve_type2(req)] if req.type=="type2" else [predict_type1(req)]
    except Exception as e:
        fb="Uncertain" if "Uncertain" in getattr(req,"options",[]) else "Unknown"
        return [{"query_id":getattr(req,"query_id","x"),"answer":fb,"unit":"","explanation":f"fallback {type(e).__name__}","premises_used":[],"reasoning":None}]
if RUN_API:
    cfg=uvicorn.Config(app,host="0.0.0.0",port=API_PORT,log_level="warning"); server=uvicorn.Server(cfg)
    threading.Thread(target=server.run,daemon=True).start(); time.sleep(5)
    print("health",requests.get(f"http://127.0.0.1:{API_PORT}/health",timeout=10).text)
if RUN_TUNNEL:
    import subprocess,pathlib,re
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cf && chmod +x /kaggle/working/cf",shell=True)
    subprocess.run("pkill -f cloudflared||true",shell=True); time.sleep(2)
    subprocess.Popen(["/kaggle/working/cf","tunnel","--url",f"http://127.0.0.1:{API_PORT}","--no-autoupdate"],stdout=open("/kaggle/working/cf.log","w"),stderr=subprocess.STDOUT)
    time.sleep(12); txt=pathlib.Path("/kaggle/working/cf.log").read_text(); m=re.search(r"https://[-a-z0-9.]+\.trycloudflare\.com",txt)
    if m:
        U=m.group(0); open("/kaggle/working/urls.txt","w").write(f"PREDICTION_URL={U}/predict\nV1_MODELS_URL={U}/v1/models\nHEALTH_URL={U}/health\n")
        print("PUBLIC:",U)